# HMCN-F Final Training

Final training of the Hierarchical Multi-Label Classification Network (HMCN-F)
for odor prediction from molecular features.

## Setup
- **Architecture**: HMCN-F with 2-level hierarchy (138 fine labels → 12 metacategories)
- **Hierarchy**: Vassilis's lookup table (126 child→parent pairs)
- **Stratification**: MultilabelStratifiedShuffleSplit on Y1 (138 fine labels)
- **Scheduler**: CosineAnnealingWarmRestarts (T0=50, Tmult=2, eta_min=1e-5)

## Best hyperparameters (from focused tuning)
| Parameter | Value |
|---|---|
| global_dim | 128 |
| local_dim | 64 |
| dropout | 0.47 |
| lr | 1e-4 |
| weight_decay | 1e-4 |
| lambda_viol | 0.1 |
| beta | 0.5 |

## Metrics computed
**Metacategories (12)**: ROC AUC, PR AUC, Balanced Accuracy, F1 (τ*), Sensitivity, Specificity, Hierarchical Violation Rate, Label Co-occurrence Consistency

**Fine labels (138)**: ROC AUC, PR AUC, F1 (τ*)

## 1. Install dependencies

In [ ]:
!pip install iterative-stratification -q

## 2. Imports and device

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix
)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Hyperparameters

In [ ]:
# Model architecture
GLOBAL_DIM  = 128
LOCAL_DIM   = 64
DROPOUT     = 0.47
BETA        = 0.5      # weight for local vs global predictions

# Training
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LAMBDA_VIOL  = 0.1     # weight for hierarchical violation penalty
BATCH_SIZE   = 32
EPOCHS       = 300
PATIENCE     = 40      # generous — warm restarts need room to recover

# Scheduler — CosineAnnealingWarmRestarts
T_0_EPOCHS = 50        # restart every 50 epochs
T_MULT     = 2         # each cycle doubles: 50 → 100 → 200
ETA_MIN    = 1e-5      # minimum lr at bottom of cosine curve

SEED = 42

## 4. Hierarchy definition (Vassilis's lookup table)

Maps each fine-grained odor descriptor to its parent metacategory.
Used to build the 126 (child, parent) pairs for the violation penalty.

In [ ]:
META_CATEGORIES = {
    'floral':        ['floral','rose','jasmin','lily','muguet','violet','hyacinth',
                      'geranium','lavender','orangeflower','chamomile','hawthorn'],
    'fruity':        ['fruity','apple','apricot','banana','berry','cherry','grape',
                      'grapefruit','lemon','melon','orange','peach','pear','pineapple',
                      'plum','raspberry','strawberry','tropical','black currant','fruit skin'],
    'sweet':         ['sweet','vanilla','caramellic','honey','chocolate','cocoa',
                      'coconut','creamy','buttery','milky','dairy'],
    'woody':         ['woody','cedar','sandalwood','pine','vetiver','terpenic',
                      'balsamic','cortex'],
    'green':         ['green','grassy','herbal','leafy','hay','tea','fresh',
                      'cucumber','vegetable','weedy'],
    'spicy':         ['spicy','cinnamon','clove','warm','pungent','sharp',
                      'cooling','mint','camphoreous'],
    'animal_musk':   ['animal','musk','leathery','fishy','sweaty','meaty',
                      'beefy','musty'],
    'earthy':        ['earthy','mushroom','nutty','hazelnut','roasted','coffee',
                      'tobacco','smoky','popcorn'],
    'citrus':        ['citrus','bergamot','ozone','clean','soapy'],
    'chemical':      ['solvent','ethereal','metallic','medicinal','phenolic',
                      'sulfurous','gassy','burnt','oily'],
    'gourmand':      ['almond','malty','rummy','brandy','cognac','winey','cooked',
                      'potato','savory','celery','tomato','radish','onion','garlic',
                      'cabbage','cheesy'],
    'powdery_amber': ['amber','powdery','anisic','coumarinic','orris','waxy',
                      'aldehydic','ketonic','lactonic'],
}

## 5. Data loading, splitting, scaling

**Stratification on Y1 (138 fine labels)**: directly protects proportions of what
the model is learning. All 138 fine labels have ≥31 positives so the algorithm
handles them cleanly (worst deviation from ideal 80/20 split: 1.25%).

**StandardScaler fitted on train only**: prevents data leakage from val/test
into the feature normalization.

In [ ]:
def load_data(csv_path='hmcn_dataset.csv'):
    df = pd.read_csv(csv_path)

    fine_cols = [c for c in df.columns if c.startswith('fine_')]
    meta_cols = [c for c in df.columns if c.startswith('meta_')]
    feat_cols = [c for c in df.columns if c not in fine_cols + meta_cols + ['SMILES']]

    # Drop zero-variance features — constant features carry no information
    stds = df[feat_cols].std()
    feat_cols = stds[stds > 0].index.tolist()

    X          = df[feat_cols].values.astype(np.float32)
    Y1         = df[fine_cols].values.astype(np.float32)   # 138 fine labels
    Y2         = df[meta_cols].values.astype(np.float32)   # 12 metacategories
    fine_names = [c.replace('fine_', '') for c in fine_cols]
    meta_names = [c.replace('meta_', '') for c in meta_cols]

    return X, Y1, Y2, fine_names, meta_names


def split_and_scale(X, Y1, Y2, seed=42):
    # Train+Val / Test — stratified on Y1
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(msss.split(X, Y1))

    # Train / Val — stratified on Y1
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=seed)
    train_idx, val_idx = next(msss2.split(X[trainval_idx], Y1[trainval_idx]))

    X_train  = X[trainval_idx][train_idx]
    X_val    = X[trainval_idx][val_idx]
    X_test   = X[test_idx]

    Y1_train = Y1[trainval_idx][train_idx]
    Y1_val   = Y1[trainval_idx][val_idx]
    Y1_test  = Y1[test_idx]

    Y2_train = Y2[trainval_idx][train_idx]
    Y2_val   = Y2[trainval_idx][val_idx]
    Y2_test  = Y2[test_idx]

    scaler   = StandardScaler()
    X_train  = scaler.fit_transform(X_train)
    X_val    = scaler.transform(X_val)
    X_test   = scaler.transform(X_test)

    return (X_train, Y1_train, Y2_train,
            X_val,   Y1_val,   Y2_val,
            X_test,  Y1_test,  Y2_test,
            scaler)


def build_violation_pairs(fine_names, meta_names):
    fine_idx = {name: i for i, name in enumerate(fine_names)}
    meta_idx = {name: i for i, name in enumerate(meta_names)}
    pairs = []
    for meta, members in META_CATEGORIES.items():
        for member in members:
            if member in fine_idx and meta in meta_idx:
                pairs.append((fine_idx[member], meta_idx[meta]))
    return pairs


# Run
X, Y1, Y2, fine_names, meta_names = load_data('hmcn_dataset.csv')
violation_pairs = build_violation_pairs(fine_names, meta_names)

(
    X_train, Y1_train, Y2_train,
    X_val,   Y1_val,   Y2_val,
    X_test,  Y1_test,  Y2_test,
    scaler
) = split_and_scale(X, Y1, Y2, seed=SEED)

print(f'Train : {len(X_train)} molecules')
print(f'Val   : {len(X_val)} molecules')
print(f'Test  : {len(X_test)} molecules')
print(f'Features        : {X_train.shape[1]}')
print(f'Fine labels     : {Y1.shape[1]}')
print(f'Meta labels     : {Y2.shape[1]}')
print(f'Hierarchy pairs : {len(violation_pairs)}')

## 6. DataLoaders

In [ ]:
def make_loader(X, Y1, Y2, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X,  dtype=torch.float32),
        torch.tensor(Y1, dtype=torch.float32),
        torch.tensor(Y2, dtype=torch.float32)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, Y1_train, Y2_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   Y1_val,   Y2_val,   batch_size=128, shuffle=False)
test_loader  = make_loader(X_test,  Y1_test,  Y2_test,  batch_size=128, shuffle=False)

## 7. Model definition

### Architecture summary

```
x (999-dim)
    │
    ├── input_proj → A_G^0 (128-dim)
    │
    ├── level1 (fine labels)
    │     [A_G^0 ⊕ x] → A_G^1, P_L1 (138-dim)
    │
    ├── level2 (metacategories)
    │     [A_G^1 ⊕ x] → A_G^2, P_L2 (12-dim)
    │
    └── global_output
          A_G^2 → P_G (150-dim)

P_F = β * [P_L1 | P_L2] + (1-β) * P_G
```

Input reuse (concatenating x at each level) lets each level access raw molecular
features directly, not just the compressed representation from previous layers.

In [ ]:
class LocalBlock(nn.Module):
    """
    One hierarchical level of HMCN-F.
    Takes the global hidden state A_G and original input x (input reuse).
    Returns the updated global state and local label predictions.
    """
    def __init__(self, input_dim, global_dim, local_dim, n_labels, dropout):
        super().__init__()

        # Global flow: (A_G ⊕ x) → next A_G
        self.global_fc = nn.Sequential(
            nn.Linear(global_dim + input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Transition: A_G → local hidden representation A_L
        self.transition = nn.Sequential(
            nn.Linear(global_dim, local_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Local output: A_L → probabilities for labels at this level
        self.output = nn.Linear(local_dim, n_labels)

    def forward(self, x, A_G):
        A_G_next = self.global_fc(torch.cat([A_G, x], dim=1))
        A_L      = self.transition(A_G_next)
        P_L      = torch.sigmoid(self.output(A_L))
        return A_G_next, P_L


class HMCNF(nn.Module):
    def __init__(self, input_dim, n_fine, n_meta,
                 global_dim, local_dim, dropout, beta):
        super().__init__()

        self.beta    = beta
        self.n_total = n_fine + n_meta

        # Initial projection: x → A_G^0
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.level1       = LocalBlock(input_dim, global_dim, local_dim, n_fine,  dropout)
        self.level2       = LocalBlock(input_dim, global_dim, local_dim, n_meta,  dropout)
        self.global_output= nn.Linear(global_dim, self.n_total)

    def forward(self, x):
        A_G       = self.input_proj(x)
        A_G, P_L1 = self.level1(x, A_G)
        A_G, P_L2 = self.level2(x, A_G)
        P_G       = torch.sigmoid(self.global_output(A_G))
        P_F       = self.beta * torch.cat([P_L1, P_L2], dim=1) + (1 - self.beta) * P_G
        return P_F, P_L1, P_L2, P_G


model = HMCNF(
    input_dim  = X_train.shape[1],
    n_fine     = Y1_train.shape[1],
    n_meta     = Y2_train.shape[1],
    global_dim = GLOBAL_DIM,
    local_dim  = LOCAL_DIM,
    dropout    = DROPOUT,
    beta       = BETA
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

## 8. Loss function

$$\mathcal{L} = \mathcal{L}_{local} + \mathcal{L}_{global} + \lambda \cdot \mathcal{L}_{viol}$$

- **Local**: BCE on fine predictions + BCE on meta predictions
- **Global**: BCE on all 150 labels from the global output head
- **Violation**: $\frac{1}{|pairs|}\sum_{(c,p)} \mathbb{E}[\max(0, P_c - P_p)^2]$ — penalizes child score exceeding parent score

In [ ]:
def binary_cross_entropy(P, Y, eps=1e-7):
    P = torch.clamp(P, eps, 1 - eps)
    return -torch.mean(Y * torch.log(P) + (1 - Y) * torch.log(1 - P))


def hierarchical_violation_penalty(P_L1, P_L2, pairs):
    total = torch.tensor(0.0, device=P_L1.device)
    for fine_idx, meta_idx in pairs:
        violation = torch.clamp(P_L1[:, fine_idx] - P_L2[:, meta_idx], min=0.0)
        total = total + torch.mean(violation ** 2)
    return total / max(len(pairs), 1)


def hmcn_loss(P_F, P_L1, P_L2, P_G, Y1, Y2, pairs, lambda_viol):
    Y_global = torch.cat([Y1, Y2], dim=1)
    local_loss     = binary_cross_entropy(P_L1, Y1) + binary_cross_entropy(P_L2, Y2)
    global_loss    = binary_cross_entropy(P_G, Y_global)
    violation_loss = hierarchical_violation_penalty(P_L1, P_L2, pairs)
    return local_loss + global_loss + lambda_viol * violation_loss

## 9. Optimizer and scheduler

**CosineAnnealingWarmRestarts**: the learning rate follows a cosine curve
that periodically resets back to lr_max. This helps escape local minima.

$$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})\left(1 + \cos\left(\frac{T_{cur}}{T_i}\pi\right)\right)$$

scheduler.step() is called **per batch** (not per epoch) for CosineAnnealingWarmRestarts.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY
)

steps_per_epoch = len(train_loader)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0     = T_0_EPOCHS * steps_per_epoch,
    T_mult  = T_MULT,
    eta_min = ETA_MIN
)

print(f'Steps per epoch : {steps_per_epoch}')
print(f'T_0 in steps    : {T_0_EPOCHS * steps_per_epoch}')

## 10. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, pairs, lambda_viol, device):
    model.train()
    total_loss = 0.0
    for X_batch, Y1_batch, Y2_batch in loader:
        X_batch  = X_batch.to(device)
        Y1_batch = Y1_batch.to(device)
        Y2_batch = Y2_batch.to(device)

        optimizer.zero_grad()
        P_F, P_L1, P_L2, P_G = model(X_batch)
        loss = hmcn_loss(P_F, P_L1, P_L2, P_G, Y1_batch, Y2_batch, pairs, lambda_viol)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()   # per-batch for CosineAnnealingWarmRestarts
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    fine_probs, fine_true = [], []
    meta_probs, meta_true = [], []
    for X_batch, Y1_batch, Y2_batch in loader:
        _, P_L1, P_L2, _ = model(X_batch.to(device))
        fine_probs.append(P_L1.cpu().numpy())
        fine_true.append(Y1_batch.numpy())
        meta_probs.append(P_L2.cpu().numpy())
        meta_true.append(Y2_batch.numpy())
    return (
        np.vstack(fine_probs), np.vstack(fine_true),
        np.vstack(meta_probs), np.vstack(meta_true)
    )


def compute_macro_roc_auc(probs, targets):
    aucs = [
        roc_auc_score(targets[:, i], probs[:, i])
        for i in range(targets.shape[1])
        if targets[:, i].sum() > 0
    ]
    return np.mean(aucs) if aucs else 0.0


def compute_macro_pr_auc(probs, targets):
    pr_aucs = [
        average_precision_score(targets[:, i], probs[:, i])
        for i in range(targets.shape[1])
        if targets[:, i].sum() > 0
    ]
    return np.mean(pr_aucs) if pr_aucs else 0.0


# Training
best_val_auc     = 0.0
best_model_state = None
patience_counter = 0
train_history    = []

print(f"{'Epoch':>5}  {'Loss':>8}  {'Val Meta AUC':>13}  {'Val Meta PRAUC':>15}  {'Val Fine AUC':>13}")
print('─' * 60)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler,
        violation_pairs, LAMBDA_VIOL, device
    )

    fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device)
    val_meta_auc    = compute_macro_roc_auc(mp_v, mt_v)
    val_meta_pr_auc = compute_macro_pr_auc(mp_v, mt_v)
    val_fine_auc    = compute_macro_roc_auc(fp_v, ft_v)

    # Current learning rate (from first param group)
    current_lr = optimizer.param_groups[0]['lr']

    train_history.append({
        'epoch'           : epoch,
        'train_loss'      : train_loss,
        'val_meta_auc'    : val_meta_auc,
        'val_meta_pr_auc' : val_meta_pr_auc,
        'val_fine_auc'    : val_fine_auc,
        'learning_rate'   : current_lr,
    })

    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:5d}  {train_loss:8.4f}  {val_meta_auc:13.4f}  {val_meta_pr_auc:15.4f}  {val_fine_auc:13.4f}")

    if val_meta_auc > best_val_auc:
        best_val_auc     = val_meta_auc
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            print(f'Best val Meta ROC AUC: {best_val_auc:.4f}')
            break


## 11. Per-label threshold search on validation set

For each label i, find τ*_i that maximises F1 on the validation set:
$$\tau_i^* = \arg\max_{\tau} F1_i(\tau)$$

**Why per-label?** Different labels have very different base rates.
A rare label like fine_chamomile (0.6% prevalence) will never exceed τ=0.5
even when correctly ranked. Per-label thresholds compensate for this.

**Note:** AUC and PR AUC are threshold-independent and unaffected by this step.

In [ ]:
def find_optimal_thresholds(probs, targets):
    candidates = np.linspace(0.05, 0.95, 19)
    thresholds = np.full(targets.shape[1], 0.5)
    for i in range(targets.shape[1]):
        if targets[:, i].sum() == 0:
            continue
        best_f1  = 0.0
        best_tau = 0.5
        for tau in candidates:
            f1 = f1_score(
                targets[:, i],
                (probs[:, i] >= tau).astype(int),
                zero_division=0
            )
            if f1 > best_f1:
                best_f1  = f1
                best_tau = tau
        thresholds[i] = best_tau
    return thresholds


# Load best checkpoint
model.load_state_dict(best_model_state)
model.to(device)

# Collect val predictions
fp_val, ft_val, mp_val, mt_val = collect_predictions(model, val_loader, device)

# Find per-label thresholds
fine_thresholds = find_optimal_thresholds(fp_val, ft_val)
meta_thresholds = find_optimal_thresholds(mp_val, mt_val)

print('Meta thresholds:')
for name, tau in zip(meta_names, meta_thresholds):
    print(f'  {name:<15}: {tau:.2f}')

## 12. Test set evaluation

### Metrics computed

**For metacategories (12) — primary comparison with BR:**
- ROC AUC, PR AUC (threshold-independent)
- Balanced Accuracy, F1, Sensitivity, Specificity (using per-label τ*)

**HMCN-specific:**
- Hierarchical violation rate
- Label co-occurrence consistency

**For fine labels (138):**
- ROC AUC, PR AUC, F1 (τ*)

## 12. Training curves

Visualize how the model learned over time:
- **Loss**: should decrease and stabilize
- **Val Meta ROC AUC**: model selection criterion — should increase then plateau
- **Val Meta PR AUC**: primary reporting metric — should follow AUC
- **Learning rate**: cosine annealing cycles with warm restarts should be visible

The vertical dashed line marks the best epoch (where model was saved).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

history_df = pd.DataFrame(train_history)
best_epoch = history_df.loc[history_df['val_meta_auc'].idxmax(), 'epoch']

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

epochs = history_df['epoch']

# ── Plot 1: Training loss ─────────────────────────────────────────────────────
ax1.plot(epochs, history_df['train_loss'], color='#2E86AB', linewidth=1.8, label='Train loss')
ax1.axvline(x=best_epoch, color='#E84855', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch ({best_epoch})')
ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Val Meta ROC AUC ──────────────────────────────────────────────────
ax2.plot(epochs, history_df['val_meta_auc'], color='#2E86AB', linewidth=1.8, label='Val Meta ROC AUC')
ax2.axvline(x=best_epoch, color='#E84855', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch ({best_epoch})')
ax2.axhline(y=history_df['val_meta_auc'].max(), color='#666', linestyle=':', linewidth=1, alpha=0.5)
ax2.set_title('Validation Meta ROC AUC\n(model selection criterion)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('ROC AUC')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Val Meta PR AUC ───────────────────────────────────────────────────
ax3.plot(epochs, history_df['val_meta_pr_auc'], color='#A23B72', linewidth=1.8, label='Val Meta PR AUC')
ax3.plot(epochs, history_df['val_fine_auc'],    color='#F18F01', linewidth=1.4,
         linestyle='--', alpha=0.8, label='Val Fine ROC AUC')
ax3.axvline(x=best_epoch, color='#E84855', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch ({best_epoch})')
ax3.set_title('Validation PR AUC (primary metric)\nvs Fine Label AUC', fontsize=13, fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('AUC')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# ── Plot 4: Learning rate schedule ───────────────────────────────────────────
ax4.plot(epochs, history_df['learning_rate'], color='#3BB273', linewidth=1.8)
ax4.axvline(x=best_epoch, color='#E84855', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch ({best_epoch})')
ax4.set_title('Learning Rate Schedule\n(CosineAnnealingWarmRestarts)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Learning Rate')
ax4.set_yscale('log')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

fig.suptitle('HMCN-F Training Curves', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('hmcn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_epoch}')
print(f'Best val Meta ROC AUC : {history_df["val_meta_auc"].max():.4f}')
print(f'Best val Meta PR AUC  : {history_df.loc[history_df["val_meta_auc"].idxmax(), "val_meta_pr_auc"]:.4f}')


In [ ]:
# Collect test predictions
fp_test, ft_test, mp_test, mt_test = collect_predictions(model, test_loader, device)


def compute_label_metrics(probs, targets, thresholds, label_names):
    """
    Compute per-label metrics matching BR baseline exactly:
    ROC AUC, PR AUC, Balanced Accuracy, F1, Sensitivity, Specificity
    """
    rows = []
    for i, name in enumerate(label_names):
        y_true = targets[:, i]
        y_prob = probs[:, i]
        y_pred = (y_prob >= thresholds[i]).astype(int)

        if y_true.sum() == 0:
            continue

        roc_auc  = roc_auc_score(y_true, y_prob)
        pr_auc   = average_precision_score(y_true, y_prob)
        bal_acc  = balanced_accuracy_score(y_true, y_pred)
        f1       = f1_score(y_true, y_pred, zero_division=0)
        sens     = recall_score(y_true, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

        rows.append({
            'label'      : name,
            'support'    : int(y_true.sum()),
            'threshold'  : round(float(thresholds[i]), 2),
            'ROC_AUC'    : round(roc_auc, 3),
            'PR_AUC'     : round(pr_auc, 3),
            'Bal_Acc'    : round(bal_acc, 3),
            'F1'         : round(f1, 3),
            'Sensitivity': round(sens, 3),
            'Specificity': round(spec, 3),
        })
    return rows


meta_rows = compute_label_metrics(mp_test, mt_test, meta_thresholds, meta_names)
fine_rows = compute_label_metrics(fp_test, ft_test, fine_thresholds, fine_names)

### 12a. Metacategory results

In [ ]:
meta_df = pd.DataFrame(meta_rows)

print('METACATEGORY RESULTS (12 labels)')
print(f"{'Label':<15} {'ROC AUC':>8} {'PR AUC':>8} {'Bal.Acc':>8} "
      f"{'F1':>6} {'Sens':>6} {'Spec':>6} {'Support':>8}")
print('─' * 75)

for _, row in meta_df.iterrows():
    print(f"{row['label']:<15} {row['ROC_AUC']:>8.3f} {row['PR_AUC']:>8.3f} "
          f"{row['Bal_Acc']:>8.3f} {row['F1']:>6.3f} "
          f"{row['Sensitivity']:>6.3f} {row['Specificity']:>6.3f} {row['support']:>8d}")

print('─' * 75)
metrics_cols = ['ROC_AUC','PR_AUC','Bal_Acc','F1','Sensitivity','Specificity']
macro = meta_df[metrics_cols].mean()
print(f"{'MACRO AVG':<15} {macro['ROC_AUC']:>8.3f} {macro['PR_AUC']:>8.3f} "
      f"{macro['Bal_Acc']:>8.3f} {macro['F1']:>6.3f} "
      f"{macro['Sensitivity']:>6.3f} {macro['Specificity']:>6.3f}")

### 12b. Hierarchical violation rate

Measures how often the model predicts a child label positive but its parent negative —
a semantically impossible prediction that BR makes freely but HMCN should avoid.

$$\text{Violation Rate} = \frac{\sum_{(c,p) \in \text{pairs}} \mathbb{1}[\hat{y}_c=1 \wedge \hat{y}_p=0]}{\sum_{(c,p) \in \text{pairs}} \mathbb{1}[\hat{y}_c=1]}$$

In [ ]:
fine_idx = {name: i for i, name in enumerate(fine_names)}
meta_idx = {name: i for i, name in enumerate(meta_names)}

# Binary predictions using per-label thresholds
fine_pred_binary = (fp_test >= fine_thresholds[np.newaxis, :]).astype(int)
meta_pred_binary = (mp_test >= meta_thresholds[np.newaxis, :]).astype(int)

total_child_positive = 0
total_violations     = 0
violation_details    = []

for meta, members in META_CATEGORIES.items():
    for member in members:
        if member not in fine_idx or meta not in meta_idx:
            continue
        fi = fine_idx[member]
        mi = meta_idx[meta]

        child_positive = fine_pred_binary[:, fi]
        parent_positive = meta_pred_binary[:, mi]

        # Violation: child=1 AND parent=0
        n_child_pos  = child_positive.sum()
        n_violations = ((child_positive == 1) & (parent_positive == 0)).sum()

        total_child_positive += n_child_pos
        total_violations     += n_violations

        if n_child_pos > 0 and n_violations > 0:
            violation_details.append({
                'child' : member,
                'parent': meta,
                'child_positive' : int(n_child_pos),
                'violations'     : int(n_violations),
                'violation_rate' : round(n_violations / n_child_pos, 3)
            })

overall_violation_rate = total_violations / total_child_positive if total_child_positive > 0 else 0

print(f'Overall hierarchical violation rate: {overall_violation_rate:.4f}')
print(f'Total child-positive predictions   : {total_child_positive}')
print(f'Total violations                   : {total_violations}')

if violation_details:
    print()
    print('Pairs with violations:')
    viol_df = pd.DataFrame(violation_details).sort_values('violation_rate', ascending=False)
    print(viol_df.to_string(index=False))
else:
    print('\nNo violations — hierarchy perfectly respected.')

### 12c. Label co-occurrence consistency

Measures whether the model has learned the co-occurrence patterns between
metacategories — e.g. that floral and sweet frequently co-occur.
BR ignores these correlations entirely; HMCN should capture them implicitly
through the shared backbone.

$$\rho_{ij} = \frac{\sum_n \hat{y}_{ni} \hat{y}_{nj}}{\sum_n \hat{y}_{ni}}$$

$$\text{Consistency} = \frac{1}{|\mathcal{P}|}\sum_{(i,j) \in \mathcal{P}} |\rho_{ij}^{pred} - \rho_{ij}^{true}|$$

Lower is better — 0 means perfect co-occurrence consistency.

In [ ]:
def compute_cooccurrence_rate(binary_preds):
    """
    Compute co-occurrence matrix: rho[i,j] = P(label_j=1 | label_i=1)
    Returns matrix of shape (n_labels, n_labels)
    """
    n_labels = binary_preds.shape[1]
    rho = np.zeros((n_labels, n_labels))
    for i in range(n_labels):
        mask = binary_preds[:, i] == 1
        if mask.sum() == 0:
            continue
        for j in range(n_labels):
            rho[i, j] = binary_preds[mask, j].mean()
    return rho


# Ground truth co-occurrence from test labels
rho_true = compute_cooccurrence_rate(mt_test.astype(int))

# Predicted co-occurrence
rho_pred = compute_cooccurrence_rate(meta_pred_binary)

# Only compute over pairs with meaningful co-occurrence in ground truth (rho > 0.1)
cooccurrence_errors = []
pair_details        = []

for i in range(len(meta_names)):
    for j in range(len(meta_names)):
        if i == j:
            continue
        if rho_true[i, j] > 0.10:   # only consider meaningful co-occurrences
            error = abs(rho_pred[i, j] - rho_true[i, j])
            cooccurrence_errors.append(error)
            pair_details.append({
                'label_i'  : meta_names[i],
                'label_j'  : meta_names[j],
                'rho_true' : round(rho_true[i, j], 3),
                'rho_pred' : round(rho_pred[i, j], 3),
                'error'    : round(error, 3)
            })

consistency_score = np.mean(cooccurrence_errors) if cooccurrence_errors else 0.0

print(f'Label co-occurrence consistency (MAE): {consistency_score:.4f}  (lower is better)')
print(f'Number of meaningful pairs evaluated : {len(cooccurrence_errors)}')
print()
print('Top 10 worst-predicted co-occurrences:')
cooc_df = pd.DataFrame(pair_details).sort_values('error', ascending=False)
print(cooc_df.head(10).to_string(index=False))

### 12d. Fine label results (top 20 by PR AUC)

In [ ]:
fine_df = pd.DataFrame(fine_rows)

print('FINE LABEL RESULTS — top 20 by PR AUC')
print(f"{'Label':<20} {'ROC AUC':>8} {'PR AUC':>8} {'F1':>6} {'Support':>8}")
print('─' * 55)

top20 = fine_df.sort_values('PR_AUC', ascending=False).head(20)
for _, row in top20.iterrows():
    print(f"{row['label']:<20} {row['ROC_AUC']:>8.3f} {row['PR_AUC']:>8.3f} "
          f"{row['F1']:>6.3f} {row['support']:>8d}")

print('─' * 55)
print(f"{'MACRO AVG (all 138)':<20} {fine_df['ROC_AUC'].mean():>8.3f} "
      f"{fine_df['PR_AUC'].mean():>8.3f} {fine_df['F1'].mean():>6.3f}")

### 12e. Full summary

### 12f. Instance-based evaluation

Label-based metrics evaluate each label independently.
Instance-based metrics evaluate the full predicted label vector per molecule —
answering: did the model correctly characterize the complete odor profile?

**Exact Match**: fraction of molecules where ALL 12 metacategory labels are correct.
$$\text{Exact Match} = \frac{1}{N}\sum_{n=1}^{N} \mathbb{1}[\hat{\mathbf{y}}_n = \mathbf{y}_n]$$

**Instance F1**: F1 computed per molecule then averaged across all molecules.
$$\text{Instance F1} = \frac{1}{N}\sum_{n=1}^{N} \frac{2|P_n \cap T_n|}{|P_n| + |T_n|}$$

Both use per-label thresholds (τ*) applied to get binary predictions.

In [ ]:
from sklearn.metrics import accuracy_score

# Binary metacategory predictions using per-label thresholds
meta_pred_binary = (mp_test >= meta_thresholds[np.newaxis, :]).astype(int)

# Exact Match — all 12 metacategory labels must be correct
exact_match = accuracy_score(mt_test, meta_pred_binary)

# Instance F1 — F1 per molecule, averaged across molecules
instance_f1 = f1_score(mt_test, meta_pred_binary, average='samples', zero_division=0)

print(f'Instance-based metrics (metacategories, per-label τ*):')
print(f'  Exact Match  : {exact_match:.4f}')
print(f'  Instance F1  : {instance_f1:.4f}')


In [ ]:
print('=' * 55)
print('FINAL RESULTS SUMMARY')
print('=' * 55)

print('\nLabel-based macro averages:')
print(f'{"Metric":<25} {"Meta (12)":>10} {"Fine (138)":>12}')
print('─' * 50)
for col, label in [('ROC_AUC','ROC AUC'), ('PR_AUC','PR AUC'),
                    ('Bal_Acc','Balanced Accuracy'),
                    ('F1','F1 (τ*)'), ('Sensitivity','Sensitivity'),
                    ('Specificity','Specificity')]:
    meta_val = meta_df[col].mean() if col in meta_df.columns else float('nan')
    fine_val = fine_df[col].mean() if col in fine_df.columns else float('nan')
    fine_str = f'{fine_val:>12.3f}' if not np.isnan(fine_val) else f'{"N/A":>12}'
    print(f'{label:<25} {meta_val:>10.3f} {fine_str}')

print(f'\nInstance-based metrics (metacategories):')
print(f'  Exact Match  : {exact_match:.4f}')
print(f'  Instance F1  : {instance_f1:.4f}')

print(f'\nHMCN-specific:')
print(f'  Hierarchical violation rate    : {overall_violation_rate:.4f}')
print(f'  Label co-occurrence consistency: {consistency_score:.4f}  (MAE, lower = better)')


## 13. Save outputs

In [ ]:
# Model weights
torch.save(best_model_state, 'hmcn_final_model.pt')

# Per-label thresholds
np.savez(
    'hmcn_final_thresholds.npz',
    fine_thresholds = fine_thresholds,
    meta_thresholds = meta_thresholds,
    fine_names      = np.array(fine_names),
    meta_names      = np.array(meta_names)
)

# All per-label metrics
all_rows = []
for row in meta_rows:
    all_rows.append({**row, 'level': 'meta'})
for row in fine_rows:
    all_rows.append({**row, 'level': 'fine'})

pd.DataFrame(all_rows).to_csv('hmcn_final_results.csv', index=False)

# Training history
pd.DataFrame(train_history).to_csv('hmcn_training_history.csv', index=False)

print('Saved:')
print('  hmcn_final_model.pt          — best model weights')
print('  hmcn_final_thresholds.npz    — per-label thresholds')
print('  hmcn_final_results.csv       — all per-label metrics (meta + fine)')
print('  hmcn_training_history.csv    — loss, AUC and LR per epoch')
print('  hmcn_training_curves.png     — training curves plot')
